In [1]:
"""

EVALUATION SCRIPT

"""

import lightning as L
from torch.utils.data import DataLoader
from datasets import load_from_disk
import torch
import os
import sys
from IKT464_AST_SSSSM import LitASR, LibriSpeechDataset, collate_fn, Config

print("Import Complete!")

/home/roberh18/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




SSM-ASR: HIERARCHICAL STATES + GATING


GPU: Tesla V100-SXM3-32GB
Memory: 31.7 GB
Compute: TF32 enabled
Vocabulary: 28 characters + blank token
Audio: 16000Hz, 80-dim log mel-spectrogram
Subsampling: 4x

Import Complete!


In [2]:
# 1. DEFINITIONS 
CKPT_PATH = "./checkpoints_v74_hier_gating_W222_D24/best_epoch=29_val_wer_clean=0.246.ckpt"
CONFIG_PATH = "./checkpoints_v74_hier_gating_W222_D24/config.json" 
DATA_PATH = "../../hub_data/librispeech"

print(f"Current Working Directory: {os.getcwd()}\n")

checks = {
    "Checkpoint": CKPT_PATH,
    "Config": CONFIG_PATH,
    "Dataset": DATA_PATH,
    "Main Script": "IKT464_AST_SSSSM.py" 
}

all_good = True

for name, path in checks.items():
    abs_path = os.path.abspath(path)
    if os.path.exists(path):
        print(f" FOUND {name}: {path}")
    else:
        print(f" MISSING {name}")
        print(f"   Looking at: {abs_path}")
        all_good = False

# --- 3. EXECUTION SETUP ---
if all_good:
    sys.path.append(os.getcwd())
    print("\nSUCCESS: All paths verified.")
else:
    raise FileNotFoundError("\nCRITICAL: One or more files are missing.")

Current Working Directory: /home/roberh18/IKT464/Medium_Subset/Exp_Depth_vs_Width

 FOUND Checkpoint: ./checkpoints_v74_hier_gating_W222_D24/best_epoch=29_val_wer_clean=0.246.ckpt
 FOUND Config: ./checkpoints_v74_hier_gating_W222_D24/config.json
 FOUND Dataset: ../../hub_data/librispeech
 FOUND Main Script: IKT464_AST_SSSSM.py

SUCCESS: All paths verified.


In [3]:
# 2. LOAD MODEL
print(f"Loading model from: {CKPT_PATH}")

model = LitASR.load_from_checkpoint(CKPT_PATH)
model.eval()

# 3. PREPARE DATA LOADERS 
dataloaders = []

print(f"Loading CLEAN data from: {DATA_PATH}")
try:
    ds_dict_clean = load_from_disk(DATA_PATH)
    test_dataset_clean = ds_dict_clean["test"]
    
    test_ds_clean = LibriSpeechDataset(
        test_dataset_clean, 
        split_name="TEST-CLEAN",
        use_specaugment=False
    )
    
    test_loader_clean = DataLoader(
        test_ds_clean, 
        batch_size=32, 
        collate_fn=collate_fn, 
        num_workers=4, 
        pin_memory=True
    )
    dataloaders.append(test_loader_clean)
    print("Successfully prepared TEST-CLEAN.")
except Exception as e:
    print(f"Error loading TEST-CLEAN: {e}")

path_other = DATA_PATH + "_other"
print(f"Loading OTHER data from: {path_other}")

try:
    if os.path.exists(path_other):
        ds_dict_other = load_from_disk(path_other)
        test_dataset_other = ds_dict_other["test"]
        
        test_ds_other = LibriSpeechDataset(
            test_dataset_other, 
            split_name="TEST-OTHER", 
            use_specaugment=False
        )
        
        test_loader_other = DataLoader(
            test_ds_other, 
            batch_size=32, 
            collate_fn=collate_fn, 
            num_workers=4, 
            pin_memory=True
        )
        dataloaders.append(test_loader_other)
        print("Successfully prepared TEST-OTHER.")
    else:
        print(f"WARNING: Path {path_other} does not exist. Skipping TEST-OTHER.")
except Exception as e:
    print(f"Error loading TEST-OTHER: {e}")

# 4. RUN INFERENCE
if dataloaders:
    # Initialize Trainer (logger=False prevents creating new log folders)
    trainer = L.Trainer(accelerator='gpu', devices=1, logger=False)
    
    print("\nRunning evaluation...")
    trainer.test(model, dataloaders=dataloaders)
else:
    print("No datasets were loaded successfully.")

Loading model from: ./checkpoints_v74_hier_gating_W222_D24/best_epoch=29_val_wer_clean=0.246.ckpt

Training Config:
   Learning rate: 0.001
   Warmup steps: 500
   Weight decay: 0.05
   SpeedPerturbation: False

SSM Encoder Configuration: d_model=222, layers=24, hierarchical=True, gating=True

Hierarchical dynamics (Target Means):
   Early (Acoustic):   a0≈0.15 (Fast decay / High Plasticity)
   Mid   (Phonetic):   a0≈0.36 (Medium decay)
   Late  (Linguistic): a0≈0.85 (Slow decay / High Stability)

  Layer  Type        a0 (mean±std)        b0 (mean±std)        c0 (mean±std)       
  ------ ----------- -------------------- -------------------- --------------------
  0      acoustic    0.147±0.085          0.350±0.049          -0.348±0.048        
  1      acoustic    0.163±0.080          0.334±0.052          -0.329±0.051        
  2      acoustic    0.177±0.080          0.320±0.049          -0.309±0.046        
  3      acoustic    0.195±0.080          0.302±0.045          -0.274±0.050  

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Dataset statistics (sampled from 1000 examples):
  Audio length: min=26480, max=527280, mean=101773
  Audio duration: min=1.7s, max=33.0s, mean=6.4s
  Text length: min=7, max=453, mean=89.6
Successfully prepared TEST-OTHER.

Running evaluation...
Testing DataLoader 1: 100%|██████████| 92/92 [01:13<00:00,  1.26it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃       DataLoader 1        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      test_wer_clean       │    0.24750611186027527    │                           │
│      test_wer_other       │                           │    0.5275760293006897     │
└───────────────────────────┴───────────────────────────┴───────────────────────────┘